<a id="ksc-overview"></a>
# 02-2. PhysicsNeMo 신경 연산자 실습 — 2D Poisson FNO

**세션:** 16:10–17:30 (신경 연산자 / FNO)  
**선행:** [01_Projectile_PINN.ipynb](01_Projectile_PINN.ipynb)

이 실습에서는 2D Poisson 방정식의 여러 소스항과 해를 학습해, 새로운 소스항이 주어졌을 때 격자 전체의 해를 예측합니다.

| 구분 | 실습 내용 |
|---|---|
| 물리 문제 | 주기적 경계조건을 갖는 2D Poisson 방정식 `-Δu=f` |
| 입력과 출력 | 소스항 `f(x,y)`를 입력하고 해 `u(x,y)`를 출력 |
| 학습 대상 | 소스항에서 해로 대응시키는 규칙 `G:f→u` |
| 모델 | 푸리에 신경 연산자(Fourier Neural Operator, FNO) |
| 구현 방식 | PhysicsNeMo-Sym의 데이터셋, 노드, `Constraint`, `Domain`, `Solver` |

수학에서는 함수 전체를 다른 함수로 대응시키는 `G`를 연산자(operator)라고 합니다. FNO는 여러 `(f,u)` 예에서 이 대응 관계를 학습합니다.

**완료 기준:** 데이터셋 검증 → 소스항과 해 한 쌍 및 푸리에 주파수 성분 확인 → FNO 구조와 설정 확인 → 학습 → 별도 테스트 데이터의 입력·정답·예측·절대 오차와 오차 지표 해석.


<a id="environment-check"></a>
## 1. 환경과 지원 파일 확인

첫 번째 코드 셀은 과정 폴더와 `../labs/poisson_fno`를 찾고, ARM64·GH200·CUDA·PhysicsNeMo 25.11 환경을 확인합니다. 필수 파일이나 GPU가 확인되지 않으면 학습을 시작하기 전에 오류 원인을 표시합니다.


In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "poisson_fno" / "generate_data.py").is_file()
        and (candidate / "labs" / "poisson_fno" / "train_fno.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "labs/poisson_fno를 찾지 못했습니다. KSC2026 과정 폴더 안에서 notebook을 열었는지 확인하세요."
    )

LAB_DIR = REPO_ROOT / "labs" / "poisson_fno"
required = [
    LAB_DIR / "data_validation.py",
    LAB_DIR / "generate_data.py",
    LAB_DIR / "train_fno.py",
    LAB_DIR / "notebook_utils.py",
    LAB_DIR / "images" / "fno_data_flow.svg",
    LAB_DIR / "conf" / "config_FNO.yaml",
    LAB_DIR / "conf" / "config_FNO_recovery.yaml",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

import torch
import physicsnemo
import physicsnemo.sym

print(f"과정 루트          : {REPO_ROOT}")
print(f"FNO 실습 폴더      : {LAB_DIR}")
print(f"Python             : {sys.version.split()[0]}")
print(f"시스템 아키텍처    : {platform.machine()}")
print(f"PyTorch            : {torch.__version__}")
print(f"PhysicsNeMo        : {getattr(physicsnemo, '__version__', '버전 정보 없음')}")
print(f"CUDA 사용 가능     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리         : {properties.total_memory / 2**30:.1f} GiB")
else:
    print("경고: CUDA GPU를 사용할 수 없습니다. 강사에게 전체 출력을 전달하세요.")


## 2. Poisson 방정식과 학습 데이터

<a id="operator-learning"></a>
단위 정사각형 `[0,1)²`에서 주기적 경계조건을 적용하고 평균값이 0인 해를 구합니다.

$$-\Delta u(x,y)=f(x,y)$$

- `f`: 방정식 오른쪽에 주어지는 소스항(source term) 또는 강제항(forcing term)
- `u`: 소스항 `f`가 주어졌을 때 구하려는 해
- `G:f→u`: 소스항 전체를 그에 대응하는 해 전체로 연결하는 연산자

주기적 경계조건에서 이 방정식이 해를 가지려면 `f`의 평균이 0이어야 합니다. 데이터 생성기는 `u`의 상수 모드를 제거한 뒤 `f=-Δu`를 계산하므로, 생성한 `u`와 `f`의 평균이 모두 0입니다.

푸리에 공간에서 `k≠0`인 모드는 다음 관계를 만족합니다.

$\hat f(k)=(2\pi)^2|k|^2\hat u(k), \qquad
\hat u(k)=\frac{\hat f(k)}{(2\pi)^2|k|^2}$

`u`에 임의의 상수를 더해도 방정식이 성립하므로, 해는 상수만큼의 차이를 제외하면 유일합니다. 이 실습은 `\hat u(0)=0`으로 두어 평균값이 0인 해를 선택합니다. `k≠0`에서 `u`의 푸리에 계수는 `f`의 푸리에 계수를 `|k|²`에 비례하는 값으로 나누어 구하므로, 높은 주파수 성분이 줄어들어 `u`가 보통 `f`보다 부드럽게 나타납니다.

Poisson 문제는 정확한 `(f,u)` 쌍을 생성하고 방정식 잔차를 검증하기 쉬워 신경 연산자의 학습 과정을 살펴보기에 적합합니다. 이 실습에서는 학습에 사용하지 않은 테스트 데이터의 오차로 FNO를 평가합니다. FFT 직접해법과의 성능 비교는 실습 범위에 포함하지 않습니다.


<a id="data-profile"></a>
## 3. 실행 설정 선택

| 실행 설정 | 용도 | 격자 | 학습 / 검증 / 테스트 데이터 수 | 데이터 `max_mode` | FNO 층 / 모드 / 채널 너비 | 배치 | 학습 단계 |
|---|---|---:|---:|---:|---:|---:|---:|
| `gh200` | GH200 대규모 후보 | 256×256 | 2048 / 256 / 256 | 24 | 6 / 32 / 64 | 8 | 2000 |
| `recovery` | 수업용 단축 실행 | 64×64 | 800 / 100 / 100 | 6 | 4 / 12 / 32 | 32 | 400 |

이 노트북은 계산량을 높인 `gh200`을 선택해 둡니다. 강사는 행사 전 PILOT GH200 예행연습에서 80분 안에 학습과 평가가 끝나는지 확인한 뒤 최종 설정을 안내합니다. `gh200` 완주를 확인하지 않은 경우에는 아래 코드 셀의 `PROFILE`을 `recovery`로 바꿉니다. 두 설정은 격자·데이터·모델·학습 단계가 모두 달라 성능 비교 대상으로 사용하지 않습니다.


In [ ]:
from notebook_utils import ensure_profile_dataset, load_field_pair, profile_info

PROFILE = "gh200"  # 행사 전 강사 안내에 따라 "recovery"로 변경
PROFILE_INFO = profile_info(LAB_DIR, PROFILE)
DATASET_DIR = PROFILE_INFO["dataset_dir"]

print(f"선택한 실행 설정 : {PROFILE}")
print(f"데이터 폴더      : {DATASET_DIR}")
print(f"격자 크기        : {PROFILE_INFO['grid_size']} × {PROFILE_INFO['grid_size']}")
print(f"데이터 max_mode  : {PROFILE_INFO['max_mode']}")
print("학습 명령        :", " ".join(PROFILE_INFO["train_command"]))


<a id="data-generation"></a>
## 4. 데이터셋 생성 및 검증

데이터 생성기는 먼저 주파수 범위를 제한하고 평균값이 0인 무작위 해 `u`를 만든 뒤, 푸리에 공간에서 `f=-Δu`를 계산합니다. 이렇게 만든 `(f,u)`는 방정식을 정확히 만족합니다. 데이터는 `f`를 입력으로, `u`를 정답으로 저장하며 FNO는 `f→u`를 학습합니다.

다음 셀은 `train.hdf5`, `val.hdf5`, `test.hdf5`에 기록된 방정식, 실행 설정, 데이터 분할, 배열 크기, Poisson 잔차 정보를 검증합니다. 모두 유효하면 기존 파일을 재사용합니다. 하나라도 유효하지 않으면 학습·검증·테스트 파일을 모두 다시 만든 뒤 재검증합니다. 데이터셋을 재사용하더라도 체크포인트와 출력 결과는 실행마다 별도 폴더에 저장합니다.


In [ ]:
FORCE_REGENERATE = False

PROFILE_INFO = ensure_profile_dataset(
    LAB_DIR,
    PROFILE,
    force=FORCE_REGENERATE,
)
DATASET_DIR = PROFILE_INFO["dataset_dir"]


<a id="data-inspection"></a>
## 5. 소스항·해 한 쌍과 푸리에 주파수 성분 확인

표본 하나는 격자에 저장된 소스항 `f`와 해 `u`의 쌍입니다. 왼쪽 두 그래프는 공간에서의 함수값을, 오른쪽 두 그래프는 푸리에 계수의 크기를 로그 눈금으로 보여 줍니다. `f`와 `u`는 값의 단위와 범위가 다를 수 있으므로 색상 눈금을 따로 사용합니다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

f_sample, u_sample = load_field_pair(LAB_DIR, PROFILE, split="test", sample_index=0)

def log_spectrum(field):
    shifted = np.fft.fftshift(np.fft.fft2(field))
    return np.log10(1.0 + np.abs(shifted))

fields = (
    f_sample,
    u_sample,
    log_spectrum(f_sample),
    log_spectrum(u_sample),
)
titles = ("f (source term)", "u (target)", "log |FFT(f)|", "log |FFT(u)|")
figure, axes = plt.subplots(1, 4, figsize=(15, 3.6), constrained_layout=True)
for axis, field, title in zip(axes, fields, titles):
    image = axis.imshow(field, origin="lower", cmap="coolwarm")
    axis.set_title(title)
    axis.set_xlabel("grid x")
    axis.set_ylabel("grid y")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
plt.show()

print(f"f 값의 범위: [{f_sample.min():.3e}, {f_sample.max():.3e}]")
print(f"u 값의 범위: [{u_sample.min():.3e}, {u_sample.max():.3e}]")


## 6. 신경 연산자 학습과 데이터 분할

격자에 기록된 소스항 `f`와 해 `u`의 쌍이 표본 하나입니다. FNO는 여러 `(f,u)` 쌍으로 학습한 뒤, 처음 보는 소스항 `f`에 대응하는 해 `u`를 예측합니다. 이 관계를 `G:f→u`로 나타냅니다.

| 데이터 분할 | 가중치 업데이트 | 역할 |
|---|---|---|
| 학습 | 사용 | 손실을 계산하고 모델의 가중치를 업데이트 |
| 검증 | 사용하지 않음 | 학습 도중 별도 데이터에서 성능 추이를 확인 |
| 테스트 | 사용하지 않음 | `Solver.solve()`가 끝난 뒤 최종 모델을 한 번 평가 |

학습·검증·테스트 데이터는 모두 같은 격자 해상도를 사용합니다. 따라서 이 실습의 일반화 범위는 **같은 해상도에서 처음 보는 소스항**입니다. 서로 다른 해상도 간 일반화와 초해상도는 후속 실험 항목입니다.


## 7. FNO 내부 구조

<p align="center"><img src="../labs/poisson_fno/images/fno_data_flow.svg" width="1050" alt="소스항에서 예측 해까지의 FNO 데이터 흐름" /></p>

<p align="center"><em>소스항 `f`를 특징 채널로 확장한 뒤 여러 푸리에 변환 블록을 거쳐 예측 해 `u_pred`를 출력합니다.</em></p>

```text
f 격자 → 특징 채널 확장 → [FFT → 선택한 모드 → 학습한 가중치 → 역 FFT
                              + 위치별 선형 변환 → 활성화] × L → 출력 변환 → u_pred 격자
```

- **특징 채널 확장(lifting):** 한 채널의 `f`를 여러 내부 특징 채널로 확장합니다.
- **주파수 경로:** FFT 뒤 설정한 수의 푸리에 모드에 학습 가능한 가중치를 적용합니다.
- **위치별 경로:** 각 공간 위치에서 선형 변환한 결과를 주파수 경로와 더합니다.
- **출력 변환(decoder):** 여러 내부 특징 채널을 한 채널의 예측 해 `u_pred`로 변환합니다.

데이터의 `max_mode`와 모델의 `fno_modes`는 역할이 다릅니다. `max_mode`는 정답 데이터에 포함되는 가장 높은 주파수를 정하고, `fno_modes`는 각 공간 축에서 푸리에 연산 층이 사용할 모드 수를 정합니다.


## 8. PhysicsNeMo-Sym 구성요소와 FNO 코드의 대응

FNO 모델은 `Create Nodes` 단계에서 생성되며, 데이터셋·학습 조건·검증기와 다음과 같이 연결됩니다.

| PhysicsNeMo-Sym 구성 | FNO 실습 코드 |
|---|---|
| Hydra 설정 | `conf/config_FNO*.yaml` 읽기 |
| Dataset | 학습·검증·테스트 HDF5 읽기 |
| Node | FNO 모델 노드 생성 |
| Constraint | `SupervisedGridConstraint`로 지도학습 손실 정의 |
| Validator | `GridValidator`로 검증 데이터의 정답과 비교 |
| Domain | Constraint와 Validator 등록 |
| Solver | 등록된 구성으로 학습 실행 |
| Geometry / Inferencer / Monitor | 이 실습에서는 사용하지 않음 |

별도 테스트 데이터 평가는 `Solver.solve()`가 끝난 뒤 최종 체크포인트를 불러와 수행합니다.


## 9. 실제 설정 파일 읽기

아래 셀은 선택한 실행 설정의 YAML 파일을 직접 읽어, 데이터 생성에 사용한 주파수 범위와 모델 규모를 나란히 표시합니다.


In [ ]:
import yaml

config_path = LAB_DIR / "conf" / f"{PROFILE_INFO['config_name']}.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

print(f"설정 파일          : {config_path.name}")
print(f"데이터 격자        : {PROFILE_INFO['grid_size']} × {PROFILE_INFO['grid_size']}")
print(f"데이터 max_mode    : {PROFILE_INFO['max_mode']}")
print(f"모델 fno_modes     : {config['arch']['fno']['fno_modes']}")
print(f"FNO 층 수          : {config['arch']['fno']['nr_fno_layers']}")
print(f"내부 채널 너비     : {config['arch']['decoder']['input_keys'][1]}")
print(f"학습 배치 크기     : {config['batch_size']['grid']}")
print(f"최대 학습 단계     : {config['training']['max_steps']}")
print(f"난수 시드          : {config['custom']['random_seed']}")


<a id="fno-level-1"></a>
## 10. FNO 학습과 체크포인트 재개

`train_fno.py`는 `DictGridDataset`, FNO 모델 노드, `SupervisedGridConstraint`, `GridValidator`, `Domain`, `Solver`를 구성합니다. 기본값은 실행 시각을 이름에 넣은 새 출력 폴더입니다.

- 처음 실행: `RESUME_RUN_DIR = None` 유지
- 중단된 실행 재개: 강사가 확인한 기존 실행 폴더를 `RESUME_RUN_DIR`에 지정

검증을 마친 데이터셋은 재사용하고, 체크포인트·검증 결과·최종 오차 지표는 실행별 폴더에 저장합니다. 코드 셀은 현재 실행의 전체 시간과 PyTorch 최대 GPU 메모리를 함께 기록합니다.


In [ ]:
from datetime import datetime
import time

RESUME_RUN_DIR = None  # 예: LAB_DIR / "outputs" / "ksc_fno_gh200" / "20260827_161015_123456"

if RESUME_RUN_DIR is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_OUTPUT_DIR = LAB_DIR / "outputs" / f"ksc_fno_{PROFILE}" / RUN_ID
    RUN_MODE = "새 학습"
else:
    RUN_OUTPUT_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    if not RUN_OUTPUT_DIR.is_dir():
        raise FileNotFoundError(f"재개할 run 폴더가 없습니다: {RUN_OUTPUT_DIR}")
    RUN_MODE = "학습 재개"

METRICS_PATH = RUN_OUTPUT_DIR / "final_state_test_metrics.json"
train_command = list(PROFILE_INFO["train_command"])
train_command.extend(
    [
        f"network_dir={RUN_OUTPUT_DIR}",
        f"custom.metrics_file={METRICS_PATH}",
    ]
)
print(f"실행 방식 : {RUN_MODE}")
print(f"결과 폴더 : {RUN_OUTPUT_DIR}")
print("실행 명령 :", " ".join(str(part) for part in train_command))

started = time.perf_counter()
completed = subprocess.run(train_command, cwd=LAB_DIR)
TRAIN_WALL_SECONDS = time.perf_counter() - started

print(f"학습 전체 시간: {TRAIN_WALL_SECONDS / 60:.2f} min")
if completed.returncode != 0:
    raise RuntimeError(f"FNO 학습 실패 (exit code={completed.returncode})")
if not METRICS_PATH.is_file():
    raise FileNotFoundError(f"평가 지표 파일을 찾지 못했습니다: {METRICS_PATH}")


<a id="result-interpretation"></a>
## 11. 평가 지표와 결과 확인

| 결과 | 의미 |
|---|---|
| 데이터셋의 최대 Poisson 잔차 | 생성한 정답 `(f,u)`가 방정식을 만족하는지 검증 |
| 검증 그래프와 손실 | 학습 도중 검증 데이터에서 모델 상태 확인 |
| 테스트 RMSE·MAE·상대 L2 오차 | 최종 모델을 별도 테스트 데이터에서 평가 |
| 예측 결과의 PDE 잔차 | 이 실습에서는 계산하지 않는 확장 평가 항목 |

아래 그림은 학습에 사용하지 않은 테스트 표본 하나의 소스항, 정답 해, 예측 해, 절대 오차를 보여 줍니다. PhysicsNeMo `Validator`의 PNG는 학습 중 검증 데이터에서 생성한 결과이고, 아래 그림은 학습 완료 후 별도 테스트 데이터에서 생성한 결과입니다. 표시한 상대 L2 오차는 전체 테스트 데이터의 L2 노름 비율입니다.


In [ ]:
import json
from IPython.display import Image, display

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
assert metrics["profile"] == PROFILE
assert metrics["problem"] == "-Delta u = f"

print(f"실행 결과 폴더      : {RUN_OUTPUT_DIR}")
print(f"학습 전체 시간      : {TRAIN_WALL_SECONDS / 60:.2f} min")
print(f"평가 상태           : {metrics['evaluation_state']}")
print(f"설정한 학습 단계    : {metrics['max_steps_configured']}")
print(f"학습 파라미터 수    : {metrics['model']['trainable_parameters']:,}")
peak_bytes = metrics["runtime_observation"]["peak_memory_allocated_bytes"]
if peak_bytes is not None:
    print(f"최대 PyTorch 메모리 : {peak_bytes / 2**30:.2f} GiB")
for name, value in metrics["metrics"].items():
    print(f"{name:20s}: {value:.6e}" if isinstance(value, float) else f"{name:20s}: {value}")

test_figure = Path(metrics["artifacts"]["held_out_test_example"]["path"])
if not test_figure.is_file():
    raise FileNotFoundError(test_figure)
display(Image(filename=str(test_figure)))


## 12. 학습 결과 정리와 실습 범위

**이번 실행에서 확인하는 항목**

- 하나의 FNO가 여러 소스항에 대응하는 해를 예측하도록 학습되는 과정
- 별도 테스트 데이터의 입력·정답·예측·절대 오차와 전체 데이터 오차 지표
- 현재 실행의 전체 소요 시간과 PyTorch가 기록한 최대 GPU 메모리
- 같은 학습 절차를 서로 다른 규모로 구성한 `gh200`·`recovery` 설정

**후속 실험 항목**

- 서로 다른 격자 해상도에서의 일반화와 초해상도
- 예측 결과의 Poisson 잔차
- FFT 직접해법과 비교한 추론 비용과 학습 비용 회수 지점
- 여러 난수 시드와 반복 실행을 사용한 성능 통계

**완료 체크리스트**

- [ ] CUDA와 데이터셋 검증을 통과했습니다.
- [ ] 학습이 종료 코드 0으로 끝나고 최종 테스트 metrics JSON이 생성됐습니다.
- [ ] 테스트 상대 L2 오차, 전체 실행 시간, 최대 PyTorch GPU 메모리를 기록했습니다.
- [ ] 소스항·정답 해·예측 해·절대 오차 그림을 해석했습니다.

필수 실습을 일찍 마친 참가자는 강사 안내 후 [FNO 푸리에 모드 수 비교](optional/FNO_Mode_Ablation.ipynb)를 진행합니다.


<a id="conceptual-extensions"></a>
<details>
<summary><strong>개념 확장 · AFNO와 PINO</strong></summary>

**AFNO(Adaptive Fourier Neural Operator)**는 푸리에 공간에서 채널을 여러 블록으로 나누어 혼합하고, soft-thresholding으로 중요도가 낮은 성분을 줄여 고해상도 공간 데이터를 효율적으로 처리하도록 설계된 모델입니다.

**PINO(Physics-Informed Neural Operator)**는 관측 또는 해 데이터에 대한 오차와 함께, PDE 잔차나 경계조건에서 계산한 물리 손실을 사용하는 신경 연산자 학습 방법입니다.

$$\mathcal{L}=\mathcal{L}_{data}+\lambda\mathcal{L}_{physics}$$

물리 손실을 사용하더라도 새로운 모든 입력에서 정확한 결과가 자동으로 보장되지는 않으므로 별도 검증이 필요합니다. 이 실습은 PhysicsNeMo-Sym으로 기본 FNO를 구성하고 평가하는 과정에 집중합니다.
</details>


---

## 참고 문헌과 라이선스

- [FNO 논문 · arXiv:2010.08895](https://arxiv.org/abs/2010.08895)
- [PhysicsNeMo 25.11 문서](https://docs.nvidia.com/physicsnemo/25.11/)
- [PhysicsNeMo-Sym 학습 조건 문서](https://docs.nvidia.com/physicsnemo/25.11/physicsnemo-sym/user_guide/features/constraints.html)

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). Existing file-level notices remain in effect.
